# Разведочный анализ подмножества HaGRIDv2 (512 px)

Используемые данные представляют собой стороннее сбалансированное подмножество HaGRIDv2: по 1 000 изображений каждой из 33 категорий жестов и категории `no_gesture`. Подмножество подходит для классификации полного кадра и детекции жестов по ограничивающим прямоугольникам (bbox).

До EDA техническая целостность проверяется отдельно из корня проекта:

```bash
python -m scripts.hagrid.validate_hagrid
```

In [ ]:
from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from PIL import Image

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError('Не найден корень проекта с data/raw')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'hagrid'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
with (DATA_ROOT / 'exported_images.json').open(encoding='utf-8') as file:
    exported = json.load(file)
with (DATA_ROOT / 'annotations.json').open(encoding='utf-8') as file:
    annotations = json.load(file)
print('HaGRID:', DATA_ROOT)

## 1. Состав и баланс классов

Цель — проверить состав используемого подмножества и определить, отражает ли баланс естественное распределение данных или процедуру отбора.

In [ ]:
class_counts = pd.Series({name: len(ids) for name, ids in exported.items()}, name='изображений').sort_index()
class_counts.index.name = 'класс'
display(class_counts.to_frame())
print('Классов:', len(class_counts), 'Изображений:', int(class_counts.sum()))
print('Минимум/максимум на класс:', int(class_counts.min()), int(class_counts.max()))

fig, ax = plt.subplots(figsize=(11, 8))
class_counts.sort_values().plot.barh(ax=ax, color='#4C78A8')
ax.set(title='Количество изображений по каталогу основного класса', xlabel='Изображений', ylabel='Класс')
ax.set_xlim(0, class_counts.max() * 1.08)
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'hagrid_class_balance.png', dpi=150)
plt.show()

## 2. Разрешения и ориентация кадров

Цель — оценить вариативность геометрии входных изображений и выбрать единый способ масштабирования с сохранением пропорций или дополнения полями. Чтение заголовков средствами Pillow не изменяет изображения.

In [ ]:
image_rows = []
for class_name, image_ids in exported.items():
    for image_id in image_ids:
        path = DATA_ROOT / class_name / f'{image_id}.jpg'
        with Image.open(path) as image:
            width, height = image.size
        orientation = 'альбомная' if width > height else ('портретная' if height > width else 'квадратная')
        image_rows.append({'class': class_name, 'image_id': image_id, 'path': path,
                           'width': width, 'height': height, 'aspect_ratio': width / height,
                           'orientation': orientation})
image_df = pd.DataFrame(image_rows)
display(image_df[['width', 'height', 'aspect_ratio']].describe().round(3))
size_counts = image_df.value_counts(['width', 'height']).rename('изображений').reset_index()
print('Различных разрешений:', len(size_counts))
display(size_counts.head(15))
orientation_counts = image_df['orientation'].value_counts()
display(orientation_counts.rename('изображений').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(size_counts['width'], size_counts['height'],
                s=np.sqrt(size_counts['изображений']) * 5, alpha=.55, color='#59A14F')
axes[0].set(title='Разрешения (размер точки ∝ частоте)', xlabel='Ширина, px', ylabel='Высота, px')
orientation_counts.plot.bar(ax=axes[1], color=['#4C78A8', '#F28E2B', '#B279A2'])
axes[1].set(title='Ориентация кадров', xlabel='', ylabel='Изображений')
axes[1].tick_params(axis='x', rotation=0)
for ax in axes: ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'hagrid_image_dimensions.png', dpi=150)
plt.show()

## 3. Полнота аннотаций

Цель — отделить фактические изображения от записей детекционной разметки и явно зафиксировать неаннотированные файлы.

In [ ]:
annotation_rows, bbox_rows = [], []
for class_name, records in annotations.items():
    for image_id, record in records.items():
        annotation_rows.append({'class': class_name, 'image_id': image_id,
                                'bbox_count': len(record['bboxes']), 'user_id': record.get('user_id')})
        for label, (x, y, width, height) in zip(record['labels'], record['bboxes']):
            bbox_rows.append({'class': class_name, 'image_id': image_id, 'label': label,
                              'x': x, 'y': y, 'width': width, 'height': height,
                              'area': width * height})
annotation_df = pd.DataFrame(annotation_rows)
bbox_df = pd.DataFrame(bbox_rows)
file_pairs = {(row['class'], row['image_id']) for row in image_rows}
annotation_pairs = set(annotation_df[['class', 'image_id']].itertuples(index=False, name=None))
missing_annotations = sorted(file_pairs - annotation_pairs)
summary = pd.Series({'изображений': len(image_df), 'аннотированных изображений': len(annotation_df),
                     'изображений без аннотации': len(missing_annotations), 'bounding boxes': len(bbox_df),
                     'записей без user_id': int(annotation_df['user_id'].isna().sum())})
display(summary.rename('значение').to_frame())
display(pd.DataFrame(missing_annotations, columns=['class', 'image_id']))

## 4. Число объектов на изображение

Цель — понять, насколько часто кадр содержит несколько размеченных рук и подходит ли упрощённая постановка «один объект на кадр».

In [ ]:
boxes_per_image = annotation_df['bbox_count'].value_counts().sort_index()
display(boxes_per_image.rename('изображений').to_frame())
print('Среднее:', round(annotation_df['bbox_count'].mean(), 4),
      'Медиана:', annotation_df['bbox_count'].median(), 'Максимум:', annotation_df['bbox_count'].max())
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(boxes_per_image.index.astype(str), boxes_per_image.values, color='#F28E2B')
ax.bar_label(bars, labels=[f'{value:,}' for value in boxes_per_image.values], padding=3)
ax.set(title='Число bounding boxes на аннотированное изображение', xlabel='Bounding boxes', ylabel='Изображений')
ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'hagrid_boxes_per_image.png', dpi=150)
plt.show()

## 5. Размеры ограничивающих прямоугольников

Цель — оценить долю кадра, занимаемую руками, и отличие `no_gesture` от целевых жестов. Это влияет на разрешение модели и стратегию аугментации.

In [ ]:
bbox_df['группа'] = np.where(bbox_df['label'].eq('no_gesture'), 'no_gesture', 'жест')
display(bbox_df.groupby('группа')[['width', 'height', 'area']].describe(
    percentiles=[.05, .25, .50, .75, .95, .99]).round(5))
bins = np.logspace(np.log10(bbox_df['area'].min()), np.log10(bbox_df['area'].max()), 60)
fig, ax = plt.subplots(figsize=(9, 5))
for label, color in [('жест', '#4C78A8'), ('no_gesture', '#E15759')]:
    ax.hist(bbox_df.loc[bbox_df['группа'].eq(label), 'area'], bins=bins, alpha=.6, label=label, color=color)
ax.set_xscale('log')
ax.set(title='Распределение площади bbox', xlabel='Доля площади кадра (логарифмическая шкала)', ylabel='Bounding boxes')
ax.legend(); ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'hagrid_bbox_areas.png', dpi=150)
plt.show()

## 6. Представленность участников

Цель — измерить повторяемость `user_id` и риск попадания изображений одного участника в обучающую, валидационную и тестовую выборки при случайном разбиении.

In [ ]:
user_counts = annotation_df['user_id'].dropna().value_counts()
display(user_counts.describe(percentiles=[.25, .50, .75, .90, .95, .99]).round(2).rename('изображений на user_id').to_frame())
missing_users_by_class = annotation_df.loc[annotation_df['user_id'].isna(), 'class'].value_counts()
display(missing_users_by_class.rename('без user_id').to_frame())
user_frequency = user_counts.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(user_frequency.index, user_frequency.values, color='#59A14F')
ax.set(title='Сколько изображений приходится на одного участника',
       xlabel='Изображений у user_id', ylabel='Уникальных user_id')
ax.set_yscale('log')
ax.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'hagrid_images_per_user.png', dpi=150)
plt.show()

## 7. Выборочная визуальная проверка разметки

Цель — проверить на примерах нескольких классов, что нормализованные координаты интерпретируются как `[x_top_left, y_top_left, width, height]`. Изображения людей используются только при интерактивном выполнении этой ячейки и не сохраняются в `reports/figures/`.

In [ ]:
example_classes = ['call', 'hand_heart', 'no_gesture']
fig, axes = plt.subplots(1, len(example_classes), figsize=(15, 5))
for ax, class_name in zip(axes, example_classes):
    image_id, record = next(iter(annotations[class_name].items()))
    with Image.open(DATA_ROOT / class_name / f'{image_id}.jpg') as source_image:
        image = source_image.convert('RGB')
    width, height = image.size
    ax.imshow(image)
    for label, (x, y, box_width, box_height) in zip(record['labels'], record['bboxes']):
        ax.add_patch(Rectangle((x * width, y * height), box_width * width, box_height * height,
                               fill=False, edgecolor='#FF3B30', linewidth=2))
        ax.text(x * width, y * height, label, color='white', fontsize=8,
                bbox={'facecolor': '#FF3B30', 'alpha': .75, 'pad': 2})
    ax.set_title(class_name); ax.axis('off')
fig.tight_layout()
plt.show()

## Выводы

- Все 34 каталога сбалансированы процедурой отбора: по 1 000 изображений на категорию. Это свойство подмножества, а не полного HaGRIDv2.
- Разрешения и ориентации различаются, хотя минимальная сторона обычно равна 512 px; масштабирование должно сохранять пропорции.
- Среди 33 991 аннотированного кадра 20 761 содержит один ограничивающий прямоугольник, 13 230 — два; постановка не сводится к одному объекту на кадр.
- Медианная площадь ограничивающего прямоугольника составляет около 1,98% кадра; в категории `no_gesture` рамки в среднем меньше, чем у целевых жестов.
- 10 586 известных участников представлены неравномерно: медиана — 1 изображение, максимум — 44. Разделение необходимо выполнять с группировкой по `user_id`.
- Девять `*-checkpoint.jpg` не имеют аннотаций, 180 записей не имеют `user_id`, десять ограничивающих прямоугольников немного выходят за границу. Правила их обработки фиксируются до подготовки выборок.